In [1]:
from vllm import LLM, SamplingParams

INFO 04-10 22:37:17 [__init__.py:239] Automatically detected platform cuda.


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [3]:
# llm = LLM(
#     model="meta-llama/Llama-3.2-3B-Instruct",
#     dtype="float16",  # if you're using fp16 to save memory
#     max_model_len=2048,  
#     gpu_memory_utilization=0.85 
# )

In [4]:
llm = LLM(
    model="./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1",
    dtype="float16",  # if you're using fp16 to save memory
    max_model_len=2048,  
    gpu_memory_utilization=0.85 
)

WARNING 04-10 22:37:17 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-10 22:37:22 [config.py:585] This model supports multiple tasks: {'embed', 'classify', 'generate', 'score', 'reward'}. Defaulting to 'generate'.
INFO 04-10 22:37:22 [llm_engine.py:241] Initializing a V0 LLM engine (v0.8.2) with config: model='./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1', speculative_config=None, tokenizer='./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar', reasoning_backend=None), observability_config=ObservabilityConfig(show_hid

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
#alpaca prompt
instruction = """Generate SVG code to visually represent the following text description, while respecting the given constraints.
        <constraints>
        * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
        * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
        </constraints>
        
        <example>
            <description>"A red circle with a blue square inside"</description>
            
            ```svg
            <svg viewBox="0 0 256 256" width="256" height="256">
              <circle cx="50" cy="50" r="40" fill="red"/>
              <rect x="30" y="30" width="40" height="40" fill="blue"/>
              <...>
               ...
              <...>
            </svg>
        ```
        </example>        
        
        Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints.
        Focus on a clear and concise representation of the input description within the given limitations. 
        Always give the complete SVG code with nothing omitted. Never use an ellipsis.
        Do not include unnecessary explanations. Just give the code.
        """
        
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
    
        ### Instruction:
        {}
    
        ### Input:             
        <description>"{}"</description>
    
        ### Response:
        """
formatted_input = alpaca_prompt.format(instruction, 'Sun rising in the East')

In [ ]:
formatted_input

In [ ]:
sampling_params = SamplingParams(temperature=0.5, top_p=0.95,max_tokens=1024)
outputs = llm.generate([formatted_input], sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(generated_text)